In [ ]:
# resume_fft_detector.py
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ds import (
    WatermarkOnTheFlyDataset,
    discover_dataset_files,
    make_train_image_augmentations,
    get_test_aug,
)
from model import make_model
from engine import (
    train_epoch,
    safe_eval_call,
    train_epoch_psnr,
    safe_eval_call_psnr,
    eval_model_psnr_with_psnr_thrs,
)
import time
from model import load_checkpoint
from datetime import timedelta
from watermark import get_watermarking_mask, get_watermarking_pattern
import warnings

warnings.filterwarnings("ignore")

# ----------------- Config (adjust if needed) -----------------
DATA_DIR = "./watermark_dataset_ring"
CHECKPOINT_PATH = "PNSR_AND_PATTERN.pth"  # or "fft_detector_ckpt_full.pth"
CKPT_NAME = "PNSR_AND_PATTERN.pth"
# CHECKPOINT_PATH = None  # or "fft_detector_ckpt_full.pth"
BATCH_SIZE = 8
NUM_WORKERS = 0
TOTAL_NUM_EPOCHS = (
    200  # total epochs you want to reach (resume will continue until this)
)
LR = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
VALIDATION_SPLIT = 0.15
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 7.5
SAVE_EVERY_EPOCHS = 1  # how often to save full checkpoint
IMAGE_SIZE = 512  # might adjust to your pipeline / VAE size
IMG_AUG = make_train_image_augmentations(IMAGE_SIZE)
TEST_AUG = get_test_aug(IMAGE_SIZE)
INCLUDE_MASK_PATCH = (
    True  # whether to include watermark mask and gt_patch in model input
)
INCLUDE_PSNR = True  # whether to include PSNR metric in dataset output

# Watermarking parameters (should match those used during watermark embedding)
W_MASK_SHAPE = "circle"
W_CHANNEL = 0
W_RADIUS = 8
W_STRENGTH = 0.9
W_PATTERN = "ring"
# -------------------------------------------------------------

# ---------------- reproducibility ----------------
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
# -------------------------------------------------

# ---------------- Load or define PIPE and TEXT_EMBEDDINGS ----------------
# Validate that PIPE and TEXT_EMBEDDINGS are present (or load them here)
try:
    import torch
    import diffusers
    from diffusers import DPMSolverMultistepScheduler
    from inverse_stable_diffusion import InversableStableDiffusionPipeline

    model_id = "stabilityai/stable-diffusion-2-1-base"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    scheduler = DPMSolverMultistepScheduler.from_pretrained(
        model_id, subfolder="scheduler"
    )
    pipe = InversableStableDiffusionPipeline.from_pretrained(
        model_id,
        scheduler=scheduler,
        torch_dtype=torch.float16,
        revision="fp16",
        verbose=False,
    )
    diffusers.utils.logging.disable_progress_bar()
    pipe.set_progress_bar_config(disable=True)
    pipe = pipe.to(device)

    TEXT_EMBEDDINGS = pipe.get_text_embedding("")  #
    PIPE = pipe  # make sure 'pipe' is in scope
except NameError:
    raise RuntimeError(
        "Please ensure `PIPE` and `TEXT_EMBEDDINGS` are available in the runtime before running resume script."
    )
# -----------------------------------------------------------------------

watermarking_mask = get_watermarking_mask(
    pipe.get_random_latents(),
    w_mask_shape=W_MASK_SHAPE,
    w_channel=W_CHANNEL,
    w_radius=W_RADIUS,
    device=device,
)

gt_patch = get_watermarking_pattern(
    pipe,
    w_seed=SEED,
    w_pattern=W_PATTERN,
    w_radius=W_RADIUS,
    device=device,
    strength=W_STRENGTH,
    shape=None,
)

# build model + optimizer + criterion
model = make_model(16, include_psnr=INCLUDE_PSNR).to(DEVICE)
model, start_epoch, best_val_loss, best_epoch = load_checkpoint(
    model, CHECKPOINT_PATH, DEVICE, opt=None
)

#  ---------------- Prepare datasets and dataloaders ----------------
file_paths, labels = discover_dataset_files(DATA_DIR)
combined = list(zip(file_paths, labels))
random.shuffle(combined)
file_paths, labels = zip(*combined)
n_val = int(len(file_paths) * VALIDATION_SPLIT)
val_paths = file_paths[:n_val]
val_labels = labels[:n_val]
print(val_labels[:50])
train_paths = file_paths[n_val:]
train_labels = labels[n_val:]

val_ds = WatermarkOnTheFlyDataset(
    val_paths,
    val_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
    image_aug=None,
    include_mask_patch=INCLUDE_MASK_PATCH,
    include_psnr=INCLUDE_PSNR,
    watermarking_mask=watermarking_mask,
    gt_patch=gt_patch,
    psnr_return_prob=False,
)
# val_ds.file_paths = val_ds.file_paths[:50]
val_ds.image_aug_prob = 1  # always apply augmentations during validation
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
crit = nn.CrossEntropyLoss()

from torchvision import transforms
from ds import RandomJPEG
from PIL import ImageFilter


# 1. No attack (identity transform / just resize to model input size)
def make_clean_aug(IMAGE_SIZE):
    return transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        ]
    )


# 2. Strong JPEG
def make_jpeg_aug(IMAGE_SIZE, q_low=40, q_high=70):
    return transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            RandomJPEG(p=1.0, q_range=(q_low, q_high)),  # always compress
        ]
    )


# 3. Blur attack
def make_blur_aug(IMAGE_SIZE):
    return transforms.Compose(
        [
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.Lambda(
                lambda img: img.filter(
                    ImageFilter.GaussianBlur(radius=random.uniform(1.5, 3.0))
                )
            ),
        ]
    )


# 4. Random crop / rotate style geometric distortion
def make_geom_aug(IMAGE_SIZE):
    return transforms.Compose(
        [
            transforms.RandomRotation(degrees=15),
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        ]
    )


def make_down_up_attack(IMAGE_SIZE, downscale_frac=0.5):
    small = max(1, int(IMAGE_SIZE * downscale_frac))
    return transforms.Compose(
        [
            transforms.Resize(
                (small, small), interpolation=transforms.InterpolationMode.BILINEAR
            ),
            transforms.Resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                interpolation=transforms.InterpolationMode.BILINEAR,
            ),
        ]
    )


def make_msg_app_combo(IMAGE_SIZE):
    # downscale -> strong jpeg -> upsample (very realistic)
    small = max(1, int(IMAGE_SIZE * 0.5))
    return transforms.Compose(
        [
            transforms.Resize(
                (small, small), interpolation=transforms.InterpolationMode.BILINEAR
            ),
            RandomJPEG(p=1.0, q_range=(40, 70)),
            transforms.Resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                interpolation=transforms.InterpolationMode.BILINEAR,
            ),
        ]
    )


def make_random_crop_attack(IMAGE_SIZE, scale=(0.5, 0.9)):
    # heavy random crop + resize (brutal for local watermarks)
    return transforms.Compose(
        [
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=scale, ratio=(0.75, 1.33)),
        ]
    )


def make_occlusion_block(IMAGE_SIZE, box_frac=0.25):
    class Block(object):
        def __init__(self, frac):
            self.frac = frac

        def __call__(self, img):
            w, h = img.size
            bw, bh = int(w * self.frac), int(h * self.frac)
            x0 = random.randint(0, max(0, w - bw))
            y0 = random.randint(0, max(0, h - bh))
            img = img.copy()
            import PIL.ImageDraw as ImageDraw

            draw = ImageDraw.Draw(img)
            draw.rectangle([x0, y0, x0 + bw, y0 + bh], fill=(0, 0, 0))
            return img

    return transforms.Compose(
        [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), Block(box_frac)]
    )


# ----------------------------------------------------------------------
# Evaluation loop (resume) with pretty printing and PSNR metrics
# ----------------------------------------------------------------------
testing_times = 5
crit = nn.CrossEntropyLoss()
total_start = time.time()


def avg(values):
    return sum(values) / len(values) if values else float("nan")


# Define attack transforms
attack_factories = {
    "clean": lambda: make_clean_aug(IMAGE_SIZE),
    "jpeg_strong": lambda: make_jpeg_aug(IMAGE_SIZE, q_low=40, q_high=60),
    "msg_app_combo": lambda: make_msg_app_combo(IMAGE_SIZE),
    "down_up": lambda: make_down_up_attack(IMAGE_SIZE, downscale_frac=0.5),
    "blur": lambda: make_blur_aug(IMAGE_SIZE),
    "random_crop": lambda: make_random_crop_attack(IMAGE_SIZE, scale=(0.5, 0.9)),
    "occlusion": lambda: make_occlusion_block(IMAGE_SIZE, box_frac=0.25),
    "geom_warp": lambda: make_geom_aug(IMAGE_SIZE),
    "train_aug_mix": lambda: make_train_image_augmentations(IMAGE_SIZE),
}

def get_best_thrs(fpr, tpr, thresholds):
    youdens_j = tpr - fpr
    best_index = np.argmax(youdens_j)
    best_thr = thresholds[best_index]
    return best_thr

# Use existing val_loader
dataset = val_loader.dataset

from psnr import (
    eval_watermark_detector_pnsr,
    batch_detector,
    eval_watermark_results,
    get_batch_results,
)

from sklearn import metrics 

all_results = {}


for attack_name, aug_builder in attack_factories.items():
    # print("\n" + "#" * 80)
    # print(f"# ATTACK: {attack_name}")
    # print("#" * 80)

    dataset.image_aug = aug_builder()  # ← change augmentation in-place
    # dataset.image_aug = IMG_AUG  # ← change augmentation in-place

    # metric trackers for this attack
    all_preds, all_gts = [], []

    for test_i in range(testing_times):
        iter_start = time.time()

        # print("=" * 80)
        # print(
        #     f"[{attack_name}] Test {test_i + 1:3d}/{testing_times:3d}    "
        #     f"Time elapsed: {str(timedelta(seconds=int(time.time() - total_start)))}"
        # )
        # print("-" * 80)

        val_loader.dataset.set_return_reversed_latents(True)
        preds, gts = eval_watermark_results(
            lambda x: get_batch_results(x, gt_patch, watermarking_mask),
            val_loader,
            device=DEVICE,
        )

        val_loader.dataset.set_return_reversed_latents(False)
        # Accumulate
        all_preds.extend(preds)
        all_gts.extend(gts)

        iter_time = time.time() - iter_start

        l1_fpr, l1_tpr, l1_thresholds = metrics.roc_curve(all_gts, [-p['l1_metric'] for p in all_preds] , pos_label=1)
        best_l1_thr = get_best_thrs(l1_fpr, l1_tpr, l1_thresholds)
        psnr_fpr, psnr_tpr, psnr_thresholds = metrics.roc_curve(all_gts, [p['psnr_metric'] for p in all_preds], pos_label=1)
        best_psnr_thr = get_best_thrs(psnr_fpr, psnr_tpr, psnr_thresholds)
        # then we decide how to do it.

    l1_fpr, l1_tpr, l1_thresholds = metrics.roc_curve(all_gts, [-p['l1_metric'] for p in all_preds] , pos_label=1)
    best_l1_thr = get_best_thrs(l1_fpr, l1_tpr, l1_thresholds)
    psnr_fpr, psnr_tpr, psnr_thresholds = metrics.roc_curve(all_gts, [p['psnr_metric'] for p in all_preds], pos_label=1)
    best_psnr_thr = get_best_thrs(psnr_fpr, psnr_tpr, psnr_thresholds)
    l1_auc = metrics.auc(l1_fpr, l1_tpr)
    psnr_auc = metrics.auc(psnr_fpr, psnr_tpr)

    # Remember the thresholds for later attack if the attack is clean
    if attack_name == "clean":
        clean_best_l1_thr = best_l1_thr
        clean_best_psnr_thr = best_psnr_thr

    # Compute final accuracy at best thresholds
    l1_preds = [1 if -p['l1_metric'] >= clean_best_l1_thr else 0 for p in all_preds]
    psnr_preds = [1 if p['psnr_metric'] >= clean_best_psnr_thr else 0 for p in all_preds]
    l1_acc = sum([1 if p == gt else 0 for p, gt in zip(l1_preds, all_gts)]) / len(all_gts)
    psnr_acc = sum([1 if p == gt else 0 for p, gt in zip(psnr_preds, all_gts)]) / len(all_gts)

    all_results[attack_name] = {
        "preds" : all_preds,
        "gts" : all_gts,
        "best_l1_thr" : best_l1_thr,
        "best_psnr_thr" : best_psnr_thr,
        "l1_acc" : l1_acc,
        "psnr_acc" : psnr_acc,
        "l1_auc" : l1_auc,
        "psnr_auc" : psnr_auc,
    }

    # Summary for this attack
    print("\n" + "-" * 80)
    print(f"AVERAGE ({attack_name}) over {testing_times} runs")
    print("-" * 80)
    print(f"{'Metric':<20} {'Avg Value':>18}")
    print("-" * 42)
    print(f"{'L1 Accuracy':<20} {l1_acc:>18.4f}")
    print(f"{'L1 AUROC':<20}   {l1_auc:>18.4f}")
    print(f"{'PSNR Accuracy':<20} {psnr_acc:>18.4f}")
    print(f"{'PSNR AUROC':<20} {psnr_auc:>18.4f}")
    print(f"{'Best L1 Thr':<20} {best_l1_thr:>18.4f}")
    print(f"{'Best PSNR Thr':<20} {best_psnr_thr:>18.4f}")
    # threshold used.
    print(f"Thresholds used for Acc: L1={clean_best_l1_thr:.4f}, PSNR={clean_best_psnr_thr:.4f}")
    print("-" * 42)
    print("#" * 80)

# save all results to a file
import pickle
with open("tree_ring_results.pkl", "wb") as f:
    pickle.dump(all_results, f)

print("\n" + "=" * 80)
print("All evaluations complete.")
print("=" * 80)

Couldn't connect to the Hub: 401 Client Error. (Request ID: Root=1-6926dbf2-0131e8a71c718313279ebbe7;4ed152f5-9cb1-4af0-a73f-a8c56d863226)

Repository Not Found for url: https://huggingface.co/api/models/stabilityai/stable-diffusion-2-1-base/revision/fp16.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password..
Will try to load from local cache.
Keyword arguments {'verbose': False} are not expected by InversableStableDiffusionPipeline and will be ignored.
Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\

Using PNSR wrapper for the model.
Loading checkpoint: PNSR_AND_PATTERN.pth
Restored model and optimizer. Resuming from epoch 133
(1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1)



--------------------------------------------------------------------------------
AVERAGE (clean) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.9759
L1 AUROC                           0.9971
PSNR Accuracy                    0.9157
PSNR AUROC                       0.9690
Best L1 Thr                    -78.2500
Best PSNR Thr                    5.3937
Thresholds used for Acc: L1=-78.2500, PSNR=5.3937
------------------------------------------
################################################################################



--------------------------------------------------------------------------------
AVERAGE (jpeg_strong) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.6627
L1 AUROC                           0.8772
PSNR Accuracy                    0.6024
PSNR AUROC                       0.7780
Best L1 Thr                    -82.2500
Best PSNR Thr                    4.9414
Thresholds used for Acc: L1=-78.2500, PSNR=5.3937
------------------------------------------
################################################################################



--------------------------------------------------------------------------------
AVERAGE (msg_app_combo) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.4747
L1 AUROC                           0.8136
PSNR Accuracy                    0.4699
PSNR AUROC                       0.7004
Best L1 Thr                    -87.3750
Best PSNR Thr                    4.3452
Thresholds used for Acc: L1=-78.2500, PSNR=5.3937
------------------------------------------
################################################################################


...


--------------------------------------------------------------------------------
AVERAGE (blur) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.4771
L1 AUROC                           0.7777
PSNR Accuracy                    0.4578
PSNR AUROC                       0.7086
Best L1 Thr                    -83.8750
Best PSNR Thr                    4.0488
Thresholds used for Acc: L1=-78.2500, PSNR=5.3937
------------------------------------------
################################################################################



--------------------------------------------------------------------------------
AVERAGE (random_crop) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.7807
L1 AUROC                           0.9565
PSNR Accuracy                    0.6795
PSNR AUROC                       0.8747
Best L1 Thr                    -82.6875
Best PSNR Thr                    5.0281
Thresholds used for Acc: L1=-78.2500, PSNR=5.3937
------------------------------------------
################################################################################



--------------------------------------------------------------------------------
AVERAGE (occlusion) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.9325
L1 AUROC                           0.9915
PSNR Accuracy                    0.8337
PSNR AUROC                       0.9426
Best L1 Thr                    -79.9375
Best PSNR Thr                    5.2857
Thresholds used for Acc: L1=-78.2500, PSNR=5.3937
------------------------------------------
################################################################################



--------------------------------------------------------------------------------
AVERAGE (geom_warp) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.7711
L1 AUROC                           0.9412
PSNR Accuracy                    0.6988
PSNR AUROC                       0.8275
Best L1 Thr                    -81.0625
Best PSNR Thr                    5.2801
Thresholds used for Acc: L1=-78.2500, PSNR=5.3937
------------------------------------------
################################################################################



--------------------------------------------------------------------------------
AVERAGE (train_aug_mix) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.6337
L1 AUROC                           0.8634
PSNR Accuracy                    0.5976
PSNR AUROC                       0.7650
Best L1 Thr                    -82.1875
Best PSNR Thr                    4.9464
Thresholds used for Acc: L1=-78.2500, PSNR=5.3937
------------------------------------------
################################################################################

All evaluations complete.


In [2]:
len(all_gts)

415

In [3]:
best_thr

NameError: name 'best_thr' is not defined

In [ ]:
l1_auc = metrics.auc(l1_fpr, l1_tpr)
psnr_auc = metrics.auc(psnr_fpr, psnr_tpr)

In [ ]:
l1_auc, psnr_auc

(0.9991085755036548, 0.9948297379211981)

In [ ]:
# Keyword arguments {'verbose': False} are not expected by InversableStableDiffusionPipeline and will be ignored.
# Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae.
# Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
# An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\unet.
# Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
# Loading pipeline components...:  60%|██████    | 3/5 [00:00<00:00, 20.70it/s]`torch_dtype` is deprecated! Use `dtype` instead!
# Loading pipeline components...: 100%|██████████| 5/5 [00:00<00:00, 12.22it/s]
# Using PNSR wrapper for the model.
# Loading checkpoint: PNSR_AND_PATTERN.pth
# Warning: couldn't fully load optimizer state: 'NoneType' object has no attribute 'load_state_dict'
# Restored model and optimizer. Resuming from epoch 133

# ################################################################################
# # ATTACK: clean
# ################################################################################
# ================================================================================
# [clean] Test   1/  5    Time elapsed: 0:00:00
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# PSNR_Acc_2                       0.9533
# PSNR_AUC_2                       0.9557
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 386.5s
# ================================================================================
# ================================================================================
# [clean] Test   2/  5    Time elapsed: 0:06:26
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# PSNR_Acc_2                       0.9533
# PSNR_AUC_2                       0.9557
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 400.4s
# ================================================================================
# ================================================================================
# [clean] Test   3/  5    Time elapsed: 0:13:07
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# PSNR_Acc_2                       0.9533
# PSNR_AUC_2                       0.9557
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 389.8s
# ================================================================================
# ================================================================================
# [clean] Test   4/  5    Time elapsed: 0:19:36
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# PSNR_Acc_2                       0.9533
# PSNR_AUC_2                       0.9557
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 382.4s
# ================================================================================
# ================================================================================
# [clean] Test   5/  5    Time elapsed: 0:25:59
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# PSNR_Acc_2                       0.9533
# PSNR_AUC_2                       0.9557
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 383.4s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (clean) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: jpeg_strong
# ################################################################################
# ================================================================================
# [jpeg_strong] Test   1/  5    Time elapsed: 0:32:22
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7200
# AUROC                              0.8272
# PSNR_Acc                         0.6067
# PSNR_AUC                         0.8568
# PSNR_Acc_2                       0.5800
# PSNR_AUC_2                       0.6013
# Val Loss                         0.5367
# -----------------------------------------
# Iter time: 382.1s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   2/  5    Time elapsed: 0:38:44
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7867
# AUROC                              0.8581
# PSNR_Acc                         0.6133
# PSNR_AUC                         0.8549
# PSNR_Acc_2                       0.6000
# PSNR_AUC_2                       0.6203
# Val Loss                         0.5246
# -----------------------------------------
# Iter time: 377.9s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   3/  5    Time elapsed: 0:45:02
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7533
# AUROC                              0.8246
# PSNR_Acc                         0.6067
# PSNR_AUC                         0.8367
# PSNR_Acc_2                       0.5933
# PSNR_AUC_2                       0.6139
# Val Loss                         0.5626
# -----------------------------------------
# Iter time: 381.3s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   4/  5    Time elapsed: 0:51:24
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7333
# AUROC                              0.8174
# PSNR_Acc                         0.5933
# PSNR_AUC                         0.8429
# PSNR_Acc_2                       0.5667
# PSNR_AUC_2                       0.5879
# Val Loss                         0.5914
# -----------------------------------------
# Iter time: 383.4s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   5/  5    Time elapsed: 0:57:47
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7400
# AUROC                              0.8214
# PSNR_Acc                         0.6133
# PSNR_AUC                         0.8529
# PSNR_Acc_2                       0.6000
# PSNR_AUC_2                       0.6203
# Val Loss                         0.5793
# -----------------------------------------
# Iter time: 378.6s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (jpeg_strong) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.7467
# AUROC                              0.8297
# PSNR_Acc                         0.6067
# PSNR_AUC                         0.8489
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: msg_app_combo
# ################################################################################
# ================================================================================
# [msg_app_combo] Test   1/  5    Time elapsed: 1:04:05
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5800
# AUROC                              0.6976
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6969
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9263
# -----------------------------------------
# Iter time: 381.9s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   2/  5    Time elapsed: 1:10:27
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5400
# AUROC                              0.6639
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6711
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9382
# -----------------------------------------
# Iter time: 384.8s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   3/  5    Time elapsed: 1:16:52
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5800
# AUROC                              0.6568
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6707
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9965
# -----------------------------------------
# Iter time: 390.1s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   4/  5    Time elapsed: 1:23:22
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5667
# AUROC                              0.7058
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6802
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9575
# -----------------------------------------
# Iter time: 382.4s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   5/  5    Time elapsed: 1:29:45
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5533
# AUROC                              0.6784
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6654
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9227
# -----------------------------------------
# Iter time: 384.3s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (msg_app_combo) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.5640
# AUROC                              0.6805
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6768
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: down_up
# ################################################################################
# ================================================================================
# [down_up] Test   1/  5    Time elapsed: 1:36:09
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 382.2s
# ================================================================================
# ================================================================================
# [down_up] Test   2/  5    Time elapsed: 1:42:31
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 380.7s
# ================================================================================
# ================================================================================
# [down_up] Test   3/  5    Time elapsed: 1:48:52
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 380.6s
# ================================================================================
# ================================================================================
# [down_up] Test   4/  5    Time elapsed: 1:55:13
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 381.1s
# ================================================================================
# ================================================================================
# [down_up] Test   5/  5    Time elapsed: 2:01:34
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 378.4s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (down_up) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: blur
# ################################################################################
# ================================================================================
# [blur] Test   1/  5    Time elapsed: 2:07:52
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6133
# AUROC                              0.6884
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.7106
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9185
# -----------------------------------------
# Iter time: 384.6s
# ================================================================================
# ================================================================================
# [blur] Test   2/  5    Time elapsed: 2:14:17
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6400
# AUROC                              0.7294
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.7427
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.8862
# -----------------------------------------
# Iter time: 392.0s
# ================================================================================
# ================================================================================
# [blur] Test   3/  5    Time elapsed: 2:20:49
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5933
# AUROC                              0.6698
# PSNR_Acc                         0.4800
# PSNR_AUC                         0.7057
# PSNR_Acc_2                       0.4800
# PSNR_AUC_2                       0.5063
# Val Loss                         0.9685
# -----------------------------------------
# Iter time: 397.7s
# ================================================================================
# ================================================================================
# [blur] Test   4/  5    Time elapsed: 2:27:26
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6400
# AUROC                              0.7602
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.7889
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.8068
# -----------------------------------------
# Iter time: 394.1s
# ================================================================================
# ================================================================================
# [blur] Test   5/  5    Time elapsed: 2:34:00
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6200
# AUROC                              0.6655
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.7073
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9983
# -----------------------------------------
# Iter time: 403.1s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (blur) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.6213
# AUROC                              0.7027
# PSNR_Acc                         0.4747
# PSNR_AUC                         0.7310
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: random_crop
# ################################################################################
# ================================================================================
# [random_crop] Test   1/  5    Time elapsed: 2:40:43
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7667
# AUROC                              0.8750
# PSNR_Acc                         0.5867
# PSNR_AUC                         0.9071
# PSNR_Acc_2                       0.5600
# PSNR_AUC_2                       0.5823
# Val Loss                         0.4597
# -----------------------------------------
# Iter time: 398.8s
# ================================================================================
# ================================================================================
# [random_crop] Test   2/  5    Time elapsed: 2:47:22
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7933
# AUROC                              0.8896
# PSNR_Acc                         0.5667
# PSNR_AUC                         0.8952
# PSNR_Acc_2                       0.5733
# PSNR_AUC_2                       0.5949
# Val Loss                         0.4375
# -----------------------------------------
# Iter time: 391.3s
# ================================================================================
# ================================================================================
# [random_crop] Test   3/  5    Time elapsed: 2:53:54
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8333
# AUROC                              0.9289
# PSNR_Acc                         0.6200
# PSNR_AUC                         0.9458
# PSNR_Acc_2                       0.5400
# PSNR_AUC_2                       0.5633
# Val Loss                         0.3648
# -----------------------------------------
# Iter time: 399.6s
# ================================================================================
# ================================================================================
# [random_crop] Test   4/  5    Time elapsed: 3:00:33
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8267
# AUROC                              0.8768
# PSNR_Acc                         0.5800
# PSNR_AUC                         0.9230
# PSNR_Acc_2                       0.6200
# PSNR_AUC_2                       0.6392
# Val Loss                         0.5174
# -----------------------------------------
# Iter time: 397.7s
# ================================================================================
# ================================================================================
# [random_crop] Test   5/  5    Time elapsed: 3:07:11
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8200
# AUROC                              0.8855
# PSNR_Acc                         0.5667
# PSNR_AUC                         0.9330
# PSNR_Acc_2                       0.5800
# PSNR_AUC_2                       0.6013
# Val Loss                         0.4716
# -----------------------------------------
# Iter time: 395.3s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (random_crop) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.8080
# AUROC                              0.8912
# PSNR_Acc                         0.5840
# PSNR_AUC                         0.9208
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: occlusion
# ################################################################################
# ================================================================================
# [occlusion] Test   1/  5    Time elapsed: 3:13:46
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9200
# AUROC                              0.9781
# PSNR_Acc                         0.9333
# PSNR_AUC                         0.9950
# PSNR_Acc_2                       0.9200
# PSNR_AUC_2                       0.9241
# Val Loss                         0.2142
# -----------------------------------------
# Iter time: 383.1s
# ================================================================================
# ================================================================================
# [occlusion] Test   2/  5    Time elapsed: 3:20:09
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9133
# AUROC                              0.9734
# PSNR_Acc                         0.9400
# PSNR_AUC                         0.9939
# PSNR_Acc_2                       0.9267
# PSNR_AUC_2                       0.9304
# Val Loss                         0.2142
# -----------------------------------------
# Iter time: 388.5s
# ================================================================================
# ================================================================================
# [occlusion] Test   3/  5    Time elapsed: 3:26:38
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9200
# AUROC                              0.9829
# PSNR_Acc                         0.9133
# PSNR_AUC                         0.9939
# PSNR_Acc_2                       0.9200
# PSNR_AUC_2                       0.9241
# Val Loss                         0.1938
# -----------------------------------------
# Iter time: 375.9s
# ================================================================================
# ================================================================================
# [occlusion] Test   4/  5    Time elapsed: 3:32:54
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9267
# AUROC                              0.9761
# PSNR_Acc                         0.9400
# PSNR_AUC                         0.9927
# PSNR_Acc_2                       0.9333
# PSNR_AUC_2                       0.9367
# Val Loss                         0.2114
# -----------------------------------------
# Iter time: 383.8s
# ================================================================================
# ================================================================================
# [occlusion] Test   5/  5    Time elapsed: 3:39:17
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9766
# PSNR_Acc                         0.9200
# PSNR_AUC                         0.9927
# PSNR_Acc_2                       0.9333
# PSNR_AUC_2                       0.9367
# Val Loss                         0.2063
# -----------------------------------------
# Iter time: 383.3s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (occlusion) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.9227
# AUROC                              0.9774
# PSNR_Acc                         0.9293
# PSNR_AUC                         0.9937
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: geom_warp
# ################################################################################
# ================================================================================
# [geom_warp] Test   1/  5    Time elapsed: 3:45:41
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8822
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.8994
# PSNR_Acc_2                       0.5333
# PSNR_AUC_2                       0.5570
# Val Loss                         0.4319
# -----------------------------------------
# Iter time: 382.8s
# ================================================================================
# ================================================================================
# [geom_warp] Test   2/  5    Time elapsed: 3:52:03
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8400
# AUROC                              0.9182
# PSNR_Acc                         0.5600
# PSNR_AUC                         0.9089
# PSNR_Acc_2                       0.5600
# PSNR_AUC_2                       0.5823
# Val Loss                         0.3926
# -----------------------------------------
# Iter time: 383.4s
# ================================================================================
# ================================================================================
# [geom_warp] Test   3/  5    Time elapsed: 3:58:27
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7867
# AUROC                              0.8763
# PSNR_Acc                         0.6000
# PSNR_AUC                         0.8863
# PSNR_Acc_2                       0.5333
# PSNR_AUC_2                       0.5570
# Val Loss                         0.4396
# -----------------------------------------
# Iter time: 385.9s
# ================================================================================
# ================================================================================
# [geom_warp] Test   4/  5    Time elapsed: 4:04:53
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7733
# AUROC                              0.8707
# PSNR_Acc                         0.5333
# PSNR_AUC                         0.8727
# PSNR_Acc_2                       0.5467
# PSNR_AUC_2                       0.5696
# Val Loss                         0.4593
# -----------------------------------------
# Iter time: 386.1s
# ================================================================================
# ================================================================================
# [geom_warp] Test   5/  5    Time elapsed: 4:11:19
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7733
# AUROC                              0.8636
# PSNR_Acc                         0.5667
# PSNR_AUC                         0.8706
# PSNR_Acc_2                       0.5733
# PSNR_AUC_2                       0.5949
# Val Loss                         0.4715
# -----------------------------------------
# Iter time: 384.3s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (geom_warp) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.7973
# AUROC                              0.8822
# PSNR_Acc                         0.5627
# PSNR_AUC                         0.8876
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: train_aug_mix
# ################################################################################
# ================================================================================
# [train_aug_mix] Test   1/  5    Time elapsed: 4:17:43
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7067
# AUROC                              0.7675
# PSNR_Acc                         0.5133
# PSNR_AUC                         0.7506
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.6584
# -----------------------------------------
# Iter time: 387.3s
# ================================================================================
# ================================================================================
# [train_aug_mix] Test   2/  5    Time elapsed: 4:24:10
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7067
# AUROC                              0.7595
# PSNR_Acc                         0.5333
# PSNR_AUC                         0.7479
# PSNR_Acc_2                       0.5200
# PSNR_AUC_2                       0.5443
# Val Loss                         0.6608
# -----------------------------------------
# Iter time: 390.2s
# ================================================================================
# ================================================================================
# [train_aug_mix] Test   3/  5    Time elapsed: 4:30:41
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7067
# AUROC                              0.8058
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.8440
# PSNR_Acc_2                       0.5733
# PSNR_AUC_2                       0.5949
# Val Loss                         0.5654
# -----------------------------------------
# Iter time: 386.5s
# ================================================================================
# ================================================================================
# [train_aug_mix] Test   4/  5    Time elapsed: 4:37:07
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7533
# AUROC                              0.8087
# PSNR_Acc                         0.5200
# PSNR_AUC                         0.7873
# PSNR_Acc_2                       0.5600
# PSNR_AUC_2                       0.5823
# Val Loss                         0.5677
# -----------------------------------------
# Iter time: 388.5s
# ================================================================================
# ================================================================================
# [train_aug_mix] Test   5/  5    Time elapsed: 4:43:36
# --------------------------------------------------------------------------------
#                                                      Metric                            Value
# -----------------------------------------
# Accuracy                         0.7667
# AUROC                              0.8214
# PSNR_Acc                         0.5667
# PSNR_AUC                         0.8540
# PSNR_Acc_2                       0.5400
# PSNR_AUC_2                       0.5633
# Val Loss                         0.5868
# -----------------------------------------
# Iter time: 381.9s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (train_aug_mix) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.7280
# AUROC                              0.7926
# PSNR_Acc                         0.5373
# PSNR_AUC                         0.7968
# ------------------------------------------
# ################################################################################

# ================================================================================
# All evaluations complete.
# ================================================================================

In [ ]:
# LR: 2.000e-04
# --------------------------------------------------------------------------------
                                                        
# Metric                            Train          Val (AUG)       Val (NO-AUG)
# --------------------------------------------------------------------------------
# Loss                             0.2427             0.3991             0.2483
# Accuracy                             --             0.8467             0.9267
# AUROC                                --             0.9162             0.9920
# --------------------------------------------------------------------------------
# Epoch time: 1289.2s (train: 925.8s). Cumulative: 10:22:51
# Best no-aug val loss so far: 0.225697 (epoch 116)
# No improvement (current no-aug val loss 0.248273)
# ================================================================================
# Epoch 132/200    Time elapsed: 10:22:51
# LR: 2.000e-04
# --------------------------------------------------------------------------------

In [ ]:
# Keyword arguments {'verbose': False} are not expected by InversableStableDiffusionPipeline and will be ignored.
# Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]`torch_dtype` is deprecated! Use `dtype` instead!
# Loading pipeline components...:  20%|██        | 1/5 [00:00<00:00,  4.91it/s]An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\unet.
# Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
# Loading pipeline components...:  60%|██████    | 3/5 [00:00<00:00,  9.89it/s]An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae.
# Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
# Loading pipeline components...: 100%|██████████| 5/5 [00:00<00:00, 13.41it/s]
# Using PNSR wrapper for the model.
# Loading checkpoint: PNSR_AND_PATTERN.pth
# Warning: couldn't fully load optimizer state: 'NoneType' object has no attribute 'load_state_dict'
# Restored model and optimizer. Resuming from epoch 133

# ################################################################################
# # ATTACK: clean
# ################################################################################
# ================================================================================
# [clean] Test   1/  5    Time elapsed: 0:00:00
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 159.6s
# ================================================================================
# ================================================================================
# [clean] Test   2/  5    Time elapsed: 0:02:39
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 161.3s
# ================================================================================
# ================================================================================
# [clean] Test   3/  5    Time elapsed: 0:05:20
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 159.0s
# ================================================================================
# ================================================================================
# [clean] Test   4/  5    Time elapsed: 0:07:59
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 165.4s
# ================================================================================
# ================================================================================
# [clean] Test   5/  5    Time elapsed: 0:10:45
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 172.6s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (clean) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: jpeg_strong
# ################################################################################
# ================================================================================
# [jpeg_strong] Test   1/  5    Time elapsed: 0:13:37
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7667
# AUROC                              0.8410
# PSNR_Acc                         0.5800
# PSNR_AUC                         0.8649
# Val Loss                         0.5454
# -----------------------------------------
# Iter time: 172.1s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   2/  5    Time elapsed: 0:16:30
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7200
# AUROC                              0.8249
# PSNR_Acc                         0.5933
# PSNR_AUC                         0.8504
# Val Loss                         0.5720
# -----------------------------------------
# Iter time: 170.8s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   3/  5    Time elapsed: 0:19:20
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7533
# AUROC                              0.8005
# PSNR_Acc                         0.6067
# PSNR_AUC                         0.8518
# Val Loss                         0.6358
# -----------------------------------------
# Iter time: 173.3s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   4/  5    Time elapsed: 0:22:14
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7467
# AUROC                              0.8315
# PSNR_Acc                         0.6000
# PSNR_AUC                         0.8299
# Val Loss                         0.5521
# -----------------------------------------
# Iter time: 177.6s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   5/  5    Time elapsed: 0:25:11
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7667
# AUROC                              0.8365
# PSNR_Acc                         0.6067
# PSNR_AUC                         0.8624
# Val Loss                         0.5804
# -----------------------------------------
# Iter time: 181.7s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (jpeg_strong) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.7507
# AUROC                              0.8269
# PSNR_Acc                         0.5973
# PSNR_AUC                         0.8519
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: msg_app_combo
# ################################################################################
# ================================================================================
# [msg_app_combo] Test   1/  5    Time elapsed: 0:28:13
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5133
# AUROC                              0.6889
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6673
# Val Loss                         0.9425
# -----------------------------------------
# Iter time: 176.9s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   2/  5    Time elapsed: 0:31:10
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5533
# AUROC                              0.6909
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6823
# Val Loss                         0.9947
# -----------------------------------------
# Iter time: 193.3s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   3/  5    Time elapsed: 0:34:23
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5733
# AUROC                              0.6987
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6903
# Val Loss                         0.9217
# -----------------------------------------
# Iter time: 191.3s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   4/  5    Time elapsed: 0:37:34
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5733
# AUROC                              0.6725
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6816
# Val Loss                         0.9867
# -----------------------------------------
# Iter time: 191.0s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   5/  5    Time elapsed: 0:40:45
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5867
# AUROC                              0.6288
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6523
# Val Loss                         1.0093
# -----------------------------------------
# Iter time: 199.7s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (msg_app_combo) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.5600
# AUROC                              0.6759
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6748
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: down_up
# ################################################################################
# ================================================================================
# [down_up] Test   1/  5    Time elapsed: 0:44:05
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 179.3s
# ================================================================================
# ================================================================================
# [down_up] Test   2/  5    Time elapsed: 0:47:04
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 164.5s
# ================================================================================
# ================================================================================
# [down_up] Test   3/  5    Time elapsed: 0:49:49
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 172.2s
# ================================================================================
# ================================================================================
# [down_up] Test   4/  5    Time elapsed: 0:52:41
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 183.3s
# ================================================================================
# ================================================================================
# [down_up] Test   5/  5    Time elapsed: 0:55:44
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 184.1s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (down_up) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: blur
# ################################################################################
# ================================================================================
# [blur] Test   1/  5    Time elapsed: 0:58:48
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5867
# AUROC                              0.7135
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.7290
# Val Loss                         0.9078
# -----------------------------------------
# Iter time: 182.8s
# ================================================================================
# ================================================================================
# [blur] Test   2/  5    Time elapsed: 1:01:51
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5533
# AUROC                              0.6326
# PSNR_Acc                         0.4800
# PSNR_AUC                         0.6907
# Val Loss                         1.0546
# -----------------------------------------
# Iter time: 184.0s
# ================================================================================
# ================================================================================
# [blur] Test   3/  5    Time elapsed: 1:04:55
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6200
# AUROC                              0.7026
# PSNR_Acc                         0.4867
# PSNR_AUC                         0.7434
# Val Loss                         0.9578
# -----------------------------------------
# Iter time: 178.4s
# ================================================================================
# ================================================================================
# [blur] Test   4/  5    Time elapsed: 1:07:54
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5733
# AUROC                              0.6630
# PSNR_Acc                         0.4933
# PSNR_AUC                         0.7074
# Val Loss                         0.9574
# -----------------------------------------
# Iter time: 175.3s
# ================================================================================
# ================================================================================
# [blur] Test   5/  5    Time elapsed: 1:10:49
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6267
# AUROC                              0.7276
# PSNR_Acc                         0.4800
# PSNR_AUC                         0.7659
# Val Loss                         0.8539
# -----------------------------------------
# Iter time: 178.6s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (blur) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.5920
# AUROC                              0.6879
# PSNR_Acc                         0.4827
# PSNR_AUC                         0.7273
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: random_crop
# ################################################################################
# ================================================================================
# [random_crop] Test   1/  5    Time elapsed: 1:13:47
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8400
# AUROC                              0.9014
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9060
# Val Loss                         0.4226
# -----------------------------------------
# Iter time: 184.2s
# ================================================================================
# ================================================================================
# [random_crop] Test   2/  5    Time elapsed: 1:16:52
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8667
# AUROC                              0.9296
# PSNR_Acc                         0.5667
# PSNR_AUC                         0.9076
# Val Loss                         0.3680
# -----------------------------------------
# Iter time: 185.1s
# ================================================================================
# ================================================================================
# [random_crop] Test   3/  5    Time elapsed: 1:19:57
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8533
# AUROC                              0.8989
# PSNR_Acc                         0.5933
# PSNR_AUC                         0.9324
# Val Loss                         0.4236
# -----------------------------------------
# Iter time: 184.4s
# ================================================================================
# ================================================================================
# [random_crop] Test   4/  5    Time elapsed: 1:23:01
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8400
# AUROC                              0.8968
# PSNR_Acc                         0.5867
# PSNR_AUC                         0.9490
# Val Loss                         0.4321
# -----------------------------------------
# Iter time: 184.2s
# ================================================================================
# ================================================================================
# [random_crop] Test   5/  5    Time elapsed: 1:26:05
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8855
# PSNR_Acc                         0.6000
# PSNR_AUC                         0.9217
# Val Loss                         0.4641
# -----------------------------------------
# Iter time: 188.1s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (random_crop) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.8427
# AUROC                              0.9024
# PSNR_Acc                         0.5800
# PSNR_AUC                         0.9234
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: occlusion
# ################################################################################
# ================================================================================
# [occlusion] Test   1/  5    Time elapsed: 1:29:13
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9467
# AUROC                              0.9809
# PSNR_Acc                         0.9133
# PSNR_AUC                         0.9957
# Val Loss                         0.1936
# -----------------------------------------
# Iter time: 178.2s
# ================================================================================
# ================================================================================
# [occlusion] Test   2/  5    Time elapsed: 1:32:12
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9400
# AUROC                              0.9786
# PSNR_Acc                         0.9200
# PSNR_AUC                         0.9955
# Val Loss                         0.2032
# -----------------------------------------
# Iter time: 171.7s
# ================================================================================
# ================================================================================
# [occlusion] Test   3/  5    Time elapsed: 1:35:03
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9267
# AUROC                              0.9761
# PSNR_Acc                         0.9333
# PSNR_AUC                         0.9925
# Val Loss                         0.2104
# -----------------------------------------
# Iter time: 171.6s
# ================================================================================
# ================================================================================
# [occlusion] Test   4/  5    Time elapsed: 1:37:55
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9200
# AUROC                              0.9766
# PSNR_Acc                         0.9400
# PSNR_AUC                         0.9943
# Val Loss                         0.2093
# -----------------------------------------
# Iter time: 170.6s
# ================================================================================
# ================================================================================
# [occlusion] Test   5/  5    Time elapsed: 1:40:46
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9200
# AUROC                              0.9791
# PSNR_Acc                         0.9200
# PSNR_AUC                         0.9914
# Val Loss                         0.2035
# -----------------------------------------
# Iter time: 173.4s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (occlusion) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.9307
# AUROC                              0.9783
# PSNR_Acc                         0.9253
# PSNR_AUC                         0.9939
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: geom_warp
# ################################################################################
# ================================================================================
# [geom_warp] Test   1/  5    Time elapsed: 1:43:39
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8200
# AUROC                              0.8734
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9009
# Val Loss                         0.4691
# -----------------------------------------
# Iter time: 173.7s
# ================================================================================
# ================================================================================
# [geom_warp] Test   2/  5    Time elapsed: 1:46:33
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7600
# AUROC                              0.8839
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.8893
# Val Loss                         0.4278
# -----------------------------------------
# Iter time: 178.0s
# ================================================================================
# ================================================================================
# [geom_warp] Test   3/  5    Time elapsed: 1:49:31
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8267
# AUROC                              0.8734
# PSNR_Acc                         0.5733
# PSNR_AUC                         0.8873
# Val Loss                         0.4509
# -----------------------------------------
# Iter time: 182.0s
# ================================================================================
# ================================================================================
# [geom_warp] Test   4/  5    Time elapsed: 1:52:33
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8067
# AUROC                              0.8659
# PSNR_Acc                         0.5467
# PSNR_AUC                         0.9123
# Val Loss                         0.4644
# -----------------------------------------
# Iter time: 183.0s
# ================================================================================
# ================================================================================
# [geom_warp] Test   5/  5    Time elapsed: 1:55:36
# --------------------------------------------------------------------------------
#                                                      Metric                            Value
# -----------------------------------------
# Accuracy                         0.8333
# AUROC                              0.9014
# PSNR_Acc                         0.5600
# PSNR_AUC                         0.9342
# Val Loss                         0.4274
# -----------------------------------------
# Iter time: 185.2s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (geom_warp) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.8093
# AUROC                              0.8796
# PSNR_Acc                         0.5573
# PSNR_AUC                         0.9048
# ------------------------------------------
# ################################################################################

# ================================================================================
# All evaluations complete.
# ================================================================================